# Paper Digest — Google Colab runner

このノートブックは GitHub の `bonsai/paper-digest` を取得し、論文PDFを解析して `outputs/latest/` に成果物を保存します。実行後、GitHub にコミットして `main` へ push します。

## 初回設定
1. このノートブックを Google Colab で開く。
2. Colab の **Secrets（鍵アイコン）** に `GITHUB_TOKEN` を登録し、ノートブックからアクセスを許可する。トークンには `bonsai/paper-digest` の Contents: Read and write 権限が必要です。
3. ランタイムで上から順に実行する。

PDFそのものは GitHub に保存しません。`outputs/latest/` は最新の全成果物、`outputs/runs/<run-id>/` は軽量な履歴スナップショットです。

In [ ]:
# Install the lightweight analysis dependencies
%pip -q install pymupdf pandas scikit-learn plotly networkx


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_DIR = Path('/content/paper-digest-src')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/bonsai/paper-digest.git', str(REPO_DIR)], check=True)

WORK_DIR = Path('/content/paper-digest-work')
OUTPUT_DIR = WORK_DIR / 'paper-digest-output'
DOWNLOAD_DIR = WORK_DIR / 'paper-digest-pdfs'
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ['PAPER_DIGEST_WORK_DIR'] = str(WORK_DIR)
os.environ['PAPER_DIGEST_OUTPUT_DIR'] = str(OUTPUT_DIR)
os.environ['PAPER_DIGEST_DOWNLOAD_DIR'] = str(DOWNLOAD_DIR)
os.environ['PAPER_DIGEST_INPUT_DIR'] = str(WORK_DIR / 'input-pdfs')
print('Repository:', REPO_DIR)
print('Output:', OUTPUT_DIR)


In [ ]:
# Run the same pipeline used by the Kaggle version
script = REPO_DIR / 'notebooks' / 'kaggle' / 'paper_digest.py'
subprocess.run([sys.executable, str(script)], check=True, env=os.environ.copy())

import json
manifest_path = OUTPUT_DIR / 'run-manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest.get('paper_count', 0) > 0, 'No papers extracted; refusing to publish.'
assert manifest.get('page_aware_chunk_count', 0) > 0, 'No text chunks extracted; refusing to publish.'
print('Validated:', manifest['paper_count'], 'papers;', manifest['page_aware_chunk_count'], 'chunks')


In [ ]:
# Publish outputs to GitHub. Token is read from Colab Secrets and never written to disk.
from google.colab import userdata
import base64, shutil
from datetime import datetime, timezone

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Colab Secrets に GITHUB_TOKEN を設定してください。')

latest = REPO_DIR / 'outputs' / 'latest'
latest.mkdir(parents=True, exist_ok=True)
for artifact in OUTPUT_DIR.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, latest / artifact.name)

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = REPO_DIR / 'outputs' / 'runs' / run_id
run_dir.mkdir(parents=True, exist_ok=True)
for name in ('run-manifest.json', 'papers.csv', 'keywords.csv'):
    source = latest / name
    if source.exists(): shutil.copy2(source, run_dir / name)

subprocess.run(['git', '-C', str(REPO_DIR), 'config', 'user.name', 'paper-digest-colab[bot]'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'config', 'user.email', 'paper-digest-colab[bot]@users.noreply.github.com'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'add', 'outputs/latest'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'add', str(Path('outputs/runs') / run_id)], check=True)
changed = subprocess.run(['git', '-C', str(REPO_DIR), 'diff', '--cached', '--quiet'])
if changed.returncode == 0:
    print('No output changes to commit.')
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'commit', '-m', f'output: publish Colab Paper Digest run {run_id}'], check=True)
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    subprocess.run(['git', '-C', str(REPO_DIR), '-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}', 'push', 'origin', 'HEAD:main'], check=True)
    print('Published outputs to GitHub:', f'https://github.com/bonsai/paper-digest/tree/main/outputs/latest')
del token, auth


## 成果物
- `outputs/latest/`: 最新実行のCSV / JSONL / HTML / manifest
- `outputs/runs/<run-id>/`: manifest、papers、keywords の軽量な履歴
- `chunks.jsonl` とグラフHTMLは最新分のみ Git に保持。元PDFは再配布せず、URLから取得します。

GitHub の `main` に直接 push するため、トークンは必ず Colab Secrets に置き、ノートブック本文や出力セルに貼り付けないでください。